<a href="https://colab.research.google.com/github/saiDan77/nn-from-scratch/blob/main/DATAPREP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced AI Data Preparation & Pipeline Engineering

---

## Module 1: Ingestion, Storage & Quality Foundations

### Chapter 01: High-Throughput Ingestion & Storage Formats

#### Core Learning Objectives
* Evaluate key storage trade-offs between row-oriented (CSV, JSON) and column-oriented (Parquet) formats.
* Implement a memory-mapped, chunked ingestion pipeline capable of streaming large files without overflowing memory.
* Validate ingested schema types dynamically before writing to persistent storage.

#### Theoretical Concept
Columnar storage (e.g., Apache Parquet) optimizes analytical workloads by storing data contiguously by column rather than row. This drastically reduces I/O via **projection pushdown** (reading only necessary columns) and enables higher compression ratios using techniques like Run-Length Encoding (RLE) and Dictionary Encoding.

In [ ]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

def stream_csv_to_parquet(csv_filepath: str, parquet_filepath: str, chunksize: int = 10_000) -> None:
    """Streams a large CSV file in chunks, enforces dynamic schema, and writes to Parquet."""
    first_chunk = True
    writer = None

    for chunk in pd.read_csv(csv_filepath, chunksize=chunksize):
        # Enforce basic numeric dynamic validation
        chunk['val'] = pd.to_numeric(chunk['val'], errors='coerce').fillna(0.0)
        table = pa.Table.from_pandas(chunk)

        if first_chunk:
            writer = pq.ParquetWriter(parquet_filepath, table.schema, compression='snappy')
            first_chunk = False

        writer.write_table(table)

    if writer:
        writer.close()

#### Failure Modes & Best Practices
* **Pitfall**: Appending directly to row-oriented structures in memory leading to Out-Of-Memory (OOM) errors.
* **Best Practice**: Stream large raw datasets using chunked iteration and store intermediate steps in columnar formats like Parquet.

---

### Chapter 02: Automated Data Quality Auditing & Schema Assertion

#### Core Learning Objectives
* Establish systematic data validation checks prior to upstream feature building.
* Implement automated missingness, bound checks, and unexpected value filtering.
* Construct actionable data quality summary matrices.

#### Theoretical Concept
Data validation defines a boundary mapping function $V: \mathcal{X} \rightarrow \{0, 1\}$ over feature space $\mathcal{X}$. Any vector $x_i \notin [\mu - k\sigma, \mu + k\sigma]$ or containing undefined types triggers assertions before entering training pipelines.

In [ ]:
import pandas as pd
import numpy as np

def validate_dataframe(df: pd.DataFrame, schema_bounds: dict) -> pd.DataFrame:
    """Audits data quality and filters records failing basic bounded schema assertions."""
    valid_mask = pd.Series(True, index=df.index)

    for col, (min_val, max_val) in schema_bounds.items():
        if col in df.columns:
            column_mask = df[col].between(min_val, max_val) | df[col].isna()
            valid_mask &= column_mask

    cleaned_df = df[valid_mask].copy()
    print(f"Audited: {len(df) - len(cleaned_df)} invalid rows dropped.")
    return cleaned_df

#### Failure Modes & Best Practices
* **Pitfall**: Silently dropping invalid records without recording schema breach metrics.
* **Best Practice**: Assert schema conditions at the ingestion layer to prevent malformed data from propagating down the MLOps pipeline.

---

## Module 2: Core Engineering, Cleaning & Feature Extraction

### Chapter 03: Missing Data Imputation & Noise Reduction

#### Core Learning Objectives
* Distinguish between Missing Completely at Random (MCAR), Missing at Random (MAR), and Missing Not at Random (MNAR).
* Implement multivariate iterative imputation techniques alongside robust signal filterings.
* Prevent data leakage by fitting transformers strictly on training sets.

#### Theoretical Concept
When data is MAR, missingness depends on observed values $Y_{obs}$. Iterative Imputation models each feature with missing values as a function of other features:
$$\hat{X}_j = f(X_{\setminus j})$$
Iteratively solving for missing values via conditional distributions prevents distortion of cross-feature correlation matrices.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

def impute_missing_multivariate(train_df: pd.DataFrame, test_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Fits an iterative multivariate imputer on train data and applies it to test data."""
    imputer = IterativeImputer(max_iter=10, random_state=42)

    train_imputed = imputer.fit_transform(train_train := train_df.select_dtypes(include=[np.number]))
    test_imputed = imputer.transform(test_df.select_dtypes(include=[np.number]))

    return (
        pd.DataFrame(train_imputed, columns=train_train.columns, index=train_df.index),
        pd.DataFrame(test_imputed, columns=train_train.columns, index=test_df.index)
    )

#### Failure Modes & Best Practices
* **Pitfall**: Imputing using global dataset metrics (mean/median) prior to train/test splits.
* **Best Practice**: Fit imputation models exclusively on training partitions, applying fitted parameters downstream.

---

### Chapter 04: Outlier Detection, Scaling & Normalization

#### Core Learning Objectives
* Detect multivariate anomalies using isolation architectures and distance thresholds.
* Implement robust scale-invariant feature transformations.
* Maintain feature distributions robust to extreme values.

#### Theoretical Concept
Standard scaling transforms inputs via:
$$z = \frac{x - \mu}{\sigma}$$
When outliers skew \mu and \sigma, **Robust Scaling** utilizes interquartile ranges (IQR):
$$x_{scaled} = \frac{x - Q_2(x)}{Q_3(x) - Q_1(x)}$$

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest

def clean_and_scale_outliers(df: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    """Detects multivariate outliers via Isolation Forest and scales robustly."""
    iso = IsolationForest(contamination=0.05, random_state=42)
    inlier_mask = iso.fit_predict(df[feature_cols]) != -1

    filtered_df = df[inlier_mask].copy()
    scaler = RobustScaler()
    filtered_df[feature_cols] = scaler.fit_transform(filtered_df[feature_cols])

    return filtered_df

#### Failure Modes & Best Practices
* **Pitfall**: Using standard z-score normalization on heavy-tailed distributions with untrimmed extreme outliers.
* **Best Practice**: Combine tree-based outlier isolation with quantile-based scaling.

---

### Chapter 05: Categorical Encoding & Dimensionality Reduction

#### Core Learning Objectives
* Contrast high-cardinality target encoding with sparse one-hot methods.
* Apply Principal Component Analysis (PCA) to reduce high-dimensional spaces while retaining target variance.
* Prevent target leakage when calculating target statistics.

#### Theoretical Concept
Target encoding replaces category $c$ with the target expectation $E[y|x=c]$. To prevent target leakage, smoothed target encoding uses a weighting factor $\lambda(n)$:
$$S_c = \lambda(n_c) \bar{y}_c + (1 - \lambda(n_c)) \bar{y}_{global}$$

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA

def encode_and_reduce(df: pd.DataFrame, cat_col: str, target_col: str, num_cols: list[str], n_components: int = 2) -> pd.DataFrame:
    """Applies smoothed target encoding and PCA dimensionality reduction."""
    # Compute smoothed mean target
    global_mean = df[target_col].mean()
    stats = df.groupby(cat_col)[target_col].agg(['count', 'mean'])
    smooth = 10
    smoothed_vals = (stats['count'] * stats['mean'] + smooth * global_mean) / (stats['count'] + smooth)

    df[f"{cat_col}_encoded"] = df[cat_col].map(smoothed_vals).fillna(global_mean)

    # Dimensionality Reduction
    pca = PCA(n_components=n_components)
    pca_features = pca.fit_transform(df[num_cols])
    for i in range(n_components):
        df[f'pca_{i}'] = pca_features[:, i]

    return df

#### Failure Modes & Best Practices
* **Pitfall**: Applying one-hot encoding directly onto continuous high-cardinality string identifiers leading to dimensional explosion.
* **Best Practice**: Use target encoding or feature embeddings for high-cardinality features.

---

### Chapter 06: Class Imbalance & Resampling Strategies

#### Core Learning Objectives
* Resolve severe class distribution imbalances in target domains.
* Implement synthetic feature generation using SMOTE alongside selective undersampling.
* Properly align evaluation metrics to non-uniform label distributions.

#### Theoretical Concept
SMOTE (Synthetic Minority Over-sampling Technique) interpolates between minority class instances. For a sample $x_i$, a random $k$-nearest neighbor $x_{zi}$ is selected, and synthetic sample $x_{new}$ is created via:
$$x_{new} = x_i + \lambda (x_{zi} - x_i) \quad \text{where } \lambda \sim U(0,1)$$

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from imblearn.over_sampling import SMOTE

def balance_dataset(X: pd.DataFrame, y: pd.Series) -> tuple[pd.DataFrame, pd.Series]:
    """Applies SMOTE to rebalance class label distributions in training partitions."""
    smote = SMOTE(random_state=42)
    X_res, y_res = smote.fit_resample(X, y)
    return pd.DataFrame(X_res, columns=X.columns), pd.Series(y_res, name=y.name)

#### Failure Modes & Best Practices
* **Pitfall**: Oversampling the target dataset prior to applying train/validation splits, leading to shared synthetic samples across partitions.
* **Best Practice**: Apply synthetic oversampling exclusively to the training split post-split.

---

## Module 3: Modality Masterclasses

### Chapter 07: Unstructured Text: Cleaning & Tokenization

#### Core Learning Objectives
* Construct robust text normalization pipelines (lowercasing, regex stripping, Unicode normalization).
* Tokenize raw natural language text using subword tokenization (WordPiece / BPE).
* Manage sequence lengths with truncation and padding masks.

#### Theoretical Concept
Subword algorithms like Byte-Pair Encoding (BPE) iteratively merge the most frequent pair of adjacent characters or tokens. The goal is to maximize corpus coverage given a fixed vocabulary size $V$, balancing out-of-vocabulary (OOV) tokens against sequence length overhead.

In [ ]:
import re
import torch
from transformers import AutoTokenizer

def prepare_text_batch(texts: list[str], model_name: str = "bert-base-uncased", max_length: int = 128) -> dict[str, torch.Tensor]:
    """Cleans text with regex and tokenizes into padded PyTorch tensors."""
    clean_texts = [re.sub(r"[^\w\s]", "", text.lower().strip()) for text in texts]

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    encoded = tokenizer(
        clean_texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    return encoded

#### Failure Modes & Best Practices
* **Pitfall**: Aggressively stripping symbols/punctuation needed for domain tasks like code generation or sentiment parsing.
* **Best Practice**: Align domain-specific regex cleaning rules with the pre-trained model's tokenization scheme.

---

### Chapter 08: Text Feature Extraction: Embeddings & Vector Stores

#### Core Learning Objectives
* Compute continuous vector representations for text blocks using pre-trained neural networks.
* Build chunking strategies optimized for downstream Retrieval-Augmented Generation (RAG).
* Index vectors for low-latency similarity queries.

#### Theoretical Concept
Cosine similarity evaluates the metric distance between dynamic sentence embeddings $u$ and $v$ within vector space $\mathbb{R}^d$:
$$\text{Sim}(u, v) = \frac{u \cdot v}{\|u\|_2 \|v\|_2}$$

In [ ]:
import numpy as np
import torch
from sklearn.metrics.pairwise import cosine_similarity

def create_embeddings_and_search(query: str, corpus: list[str], embed_fn) -> list[tuple[str, float]]:
    """Generates embeddings for corpus chunks and ranks them by cosine similarity."""
    corpus_embeddings = embed_fn(corpus)  # Expected shape: (N, D)
    query_embedding = embed_fn([query])   # Expected shape: (1, D)

    scores = cosine_similarity(query_embedding, corpus_embeddings)[0]
    ranked_indices = np.argsort(scores)[::-1]

    return [(corpus[idx], float(scores[idx])) for idx in ranked_indices]

#### Failure Modes & Best Practices
* **Pitfall**: Naively chunking documents purely by character count without taking structural boundaries (paragraphs, headings) into account.
* **Best Practice**: Use semantic or sentence-boundary chunking algorithms for vector store indexing.

---

### Chapter 09: Computer Vision: Preprocessing, Augmentation & Spatial Transforms

#### Core Learning Objectives
* Perform standardized spatial transforms, resizings, and color channel normalizations.
* Build online dataset augmentation pipelines to maximize variance during model training.
* Validate bounding boxes and mask transformations during visual geometry edits.

#### Theoretical Concept
Image normalization maps raw pixel intensities $P_{i,j,c} \in [0, 255]$ into standardized continuous values using channel-wise parameters:
$$\hat{P}_{i,j,c} = \frac{\frac{P_{i,j,c}}{255} - \mu_c}{\sigma_c}$$

In [ ]:
import torch
from torchvision import transforms
from PIL import Image

def get_vision_transform_pipeline(image_size: tuple[int, int] = (224, 224)) -> transforms.Compose:
    """Builds a image preprocessing and augmentation pipeline."""
    return transforms.Compose([
        transforms.Resize(image_size),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

#### Failure Modes & Best Practices
* **Pitfall**: Applying spatial transforms (e.g., cropping/flipping) to input images without adjusting corresponding object detection bounding boxes.
* **Best Practice**: Ensure visual augmentation libraries synchronized spatial transforms across image inputs and annotation layers.

---

### Chapter 10: Audio Preprocessing: Spectrograms & Feature Extraction

#### Core Learning Objectives
* Convert raw temporal audio signals into time-frequency representations.
* Compute Mel-Frequency Cepstral Coefficients (MFCCs) for speech processing workloads.
* Apply temporal trimming, padding, and gain normalizations to raw waveforms.

#### Theoretical Concept
The Short-Time Fourier Transform (STFT) computes discrete Fourier transforms over overlapping windowed segments of a continuous signal $x[n]$:
$$X(m, \omega) = \sum_{n=-\infty}^{\infty} x[n] w[n-m] e^{-j\omega n}$$
Mapping $|X(m, \omega)|^2$ to non-linear Mel filterbanks models human auditory perception.

In [ ]:
import numpy as np
import torch

def compute_mel_spectrogram(waveform: torch.Tensor, sample_rate: int = 16000, n_mels: int = 64) -> torch.Tensor:
    """Computes a normalized Mel-Spectrogram tensor from raw 1D audio waveforms."""
    # Ensure standard length
    max_len = sample_rate * 3 # 3 seconds
    if waveform.shape[-1] > max_len:
        waveform = waveform[..., :max_len]
    else:
        waveform = torch.nn.functional.pad(waveform, (0, max_len - waveform.shape[-1]))

    # Standard STFT signal transform placeholder representation via matrix operations
    stft = torch.stft(waveform, n_fft=400, hop_length=160, return_complex=True)
    spectrogram = torch.abs(stft)**2
    return spectrogram

#### Failure Modes & Best Practices
* **Pitfall**: Mixing audio inputs recorded at different sampling rates without standardizing resampling upfront.
* **Best Practice**: Convert all incoming audio assets to a single target sampling rate (e.g., 16kHz) early in ingestion.

---

### Chapter 11: Time Series: Feature Engineering & Windowing

#### Core Learning Objectives
* Convert continuous sequence streams into tabular sliding-window datasets.
* Engineer stationary time-series inputs using lag variables, rolling statistics, and differencing.
* Handle irregular temporal sampling and missing timestamp entries.

#### Theoretical Concept
To remove non-stationarity driven by trend/seasonality, first-order differencing is applied:
$$\Delta Y_t = Y_t - Y_{t-1}$$
Sliding window mappings construct target matrices $Y \in \mathbb{R}^{N \times h}$ from historical lags $X \in \mathbb{R}^{N \times w}$.

In [ ]:
import pandas as pd
import numpy as np

def create_sliding_windows(df: pd.DataFrame, target_col: str, window_size: int = 5, horizon: int = 1) -> tuple[np.ndarray, np.ndarray]:
    """Converts a univariate time series dataframe into sliding input/output arrays."""
    series = df[target_col].values
    X, y = [], []
    for i in range(len(series) - window_size - horizon + 1):
        X.append(series[i : i + window_size])
        y.append(series[i + window_size : i + window_size + horizon])
    return np.array(X), np.array(y)

#### Failure Modes & Best Practices
* **Pitfall**: Computing global rolling aggregates using future values (look-ahead bias / data leakage).
* **Best Practice**: Use strict trailing/backward windows when calculating moving averages or scaling temporal features.

---

### Chapter 12: Multimodal Processing & Multi-Tensor Fusion

#### Core Learning Objectives
* Align disparate modalities (e.g., text, images, tabular) along shared batch dimensions.
* Construct custom PyTorch `Dataset` structures designed for joint multi-tensor streaming.
* Apply cross-modal missing value masks during batch assembly.

#### Theoretical Concept
Multimodal fusion integrates distinct representation spaces $H_A \in \mathbb{R}^{d_A}$ and $H_B \in \mathbb{R}^{d_B}$. Late/Cross-Attention concatenation constructs a unified tensor space $H_{fused}$:
$$H_{fused} = [W_A H_A \,||\, W_B H_B] \quad \text{where } W_k \in \mathbb{R}^{d_{target} \times d_k}$$

In [ ]:
import torch
from torch.utils.data import Dataset

class MultimodalDataset(Dataset):
    """Custom dataset handling unified aligned indexing across text and image tensors."""
    def __init__(self, text_tensors: torch.Tensor, image_tensors: torch.Tensor, labels: torch.Tensor):
        assert len(text_tensors) == len(image_tensors) == len(labels)
        self.text = text_tensors
        self.images = image_tensors
        self.labels = labels

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        return {
            "text_inputs": self.text[idx],
            "image_inputs": self.images[idx],
            "label": self.labels[idx]
        }

#### Failure Modes & Best Practices
* **Pitfall**: Failing to handle unaligned indexing across modalities, leading to cross-contamination of targets and features.
* **Best Practice**: Enforce key verification across all input sources prior to constructing multi-tensor datasets.

---

## Module 4: Production, MLOps & Advanced AI Pipeline Engineering

### Chapter 13: Feature Stores & Real-Time Pipeline Optimization

#### Core Learning Objectives
* Bridge operational offline batch features with real-time low-latency online serving structures.
* Construct dual-storage feature definitions.
* Prevent train/serve skew across pipeline execution environments.

#### Theoretical Concept
A **Feature Store** manages two data engines: an **Offline Store** (e.g., Parquet/S3) optimized for high-throughput historical queries, and an **Online Store** (e.g., key-value caches like Redis) optimized for low-latency point lookups:
$$\text{Latency}_{\text{online}} \ll 10\text{ms}$$

In [ ]:
import time
import pandas as pd

class MockOnlineFeatureStore:
    """Simulates an online feature store reading from an in-memory key-value cache."""
    def __init__(self):
        self._store = {}

    def push_features(self, entity_id: str, feature_dict: dict) -> None:
        """Writes entity features to the online store."""
        self._store[entity_id] = {**feature_dict, "_timestamp": time.time()}

    def get_online_features(self, entity_ids: list[str]) -> list[dict]:
        """Fetches features for low-latency online inference."""
        return [self._store.get(eid, {}) for eid in entity_ids]

#### Failure Modes & Best Practices
* **Pitfall**: Re-implementing feature transformations separately in online vs. offline environments, leading to serving skew.
* **Best Practice**: Define transformation logic once in a central feature definition store.

---

### Chapter 14: Data Drift, Covariate Shift & Monitoring

#### Core Learning Objectives
* Measure distribution drift between reference training sets and incoming production inferences.
* Implement Statistical tests (KS-Test, Population Stability Index) to alert engineers to drift.
* Construct operational triggers for automated pipeline retraining.

#### Theoretical Concept
The **Population Stability Index (PSI)** quantifies variations between reference distribution $P$ and actual target distribution $Q$ across $k$ bins:
$$\text{PSI} = \sum_{b=1}^{k} \left( Q_b - P_b \right) \times \ln\left(\frac{Q_b}{P_b}\right)$$
$\text{PSI} > 0.2$ indicates significant distribution drift.

In [ ]:
import numpy as np
from scipy.stats import ks_2samp

def detect_covariate_shift(reference_data: np.ndarray, current_data: np.ndarray, alpha: float = 0.05) -> bool:
    """Uses a two-sample Kolmogorov-Smirnov test to detect feature distribution drift."""
    ks_stat, p_value = ks_2samp(reference_data, current_data)
    drift_detected = p_value < alpha
    print(f"KS Statistic: {ks_stat:.4f} | p-value: {p_value:.4f} | Drift Detected: {drift_detected}")
    return drift_detected

#### Failure Modes & Best Practices
* **Pitfall**: Monitoring model evaluation outputs exclusively while ignoring silent upstream feature drift.
* **Best Practice**: Run statistical validation checks on key incoming raw features continuously.

---

### Chapter 15: Privacy, Anonymization & Governance

#### Core Learning Objectives
* Anonymize Sensitive Personally Identifiable Information (PII) features using irreversible hashing/masking.
* Apply Differential Privacy (\epsilon, \delta) guarantees to statistical analytics outputs.
* Track end-to-end data lineage across processing jobs.

#### Theoretical Concept
A randomized mechanism $M$ satisfies $(\epsilon, \delta)$**-Differential Privacy** if for all neighboring datasets $D_1, D_2$ differing on a single record, and all query outputs $S \subseteq \text{Range}(M)$:
$$P[M(D_1) \in S] \le e^\epsilon \cdot P[M(D_2) \in S] + \delta$$

In [ ]:
import hashlib
import pandas as pd

def anonymize_pii_dataframe(df: pd.DataFrame, pii_cols: list[str], salt: str = "secure_salt") -> pd.DataFrame:
    """Hashes PII columns using SHA-256 with a secure salt."""
    anonymized_df = df.copy()
    for col in pii_cols:
        anonymized_df[col] = anonymized_df[col].astype(str).apply(
            lambda val: hashlib.sha256((val + salt).encode('utf-8')).hexdigest()
        )
    return anonymized_df

#### Failure Modes & Best Practices
* **Pitfall**: Hashing sensitive identifying attributes without a salt, leaving strings vulnerable to dictionary attack lookups.
* **Best Practice**: Combine cryptographic salting with strict role-based access control (RBAC) governance.

---

### Chapter 16: Synthetic Data Generation & Privacy-Preserving Augmentation

#### Core Learning Objectives
* Generate privacy-preserving synthetic tabular inputs using generative models or probabilistic distributions.
* Preserve cross-feature covariance structures without exposing original sensitive records.
* Evaluate statistical fidelity between real and generated distributions.

#### Theoretical Concept
Generative models sample from a learned continuous probability density function $p_\theta(x) \approx p_{data}(x)$. Sampling $\hat{x} \sim p_\theta(x)$ creates new instances that mimic the underlying distribution without duplicating original training records.

In [ ]:
import numpy as np
import pandas as pd

def generate_synthetic_gaussian_data(reference_df: pd.DataFrame, num_samples: int = 100) -> pd.DataFrame:
    """Generates synthetic tabular features using multivariate Gaussian sampling."""
    means = reference_df.mean()
    cov_matrix = reference_df.cov()

    synthetic_raw = np.random.multivariate_normal(mean=means, cov=cov_matrix, size=num_samples)
    return pd.DataFrame(synthetic_raw, columns=reference_df.columns)

#### Failure Modes & Best Practices
* **Pitfall**: Evaluating synthetic data solely on univariate distributions while ignoring distorted inter-feature correlations.
* **Best Practice**: Validate synthetic outputs using multivariate correlation matrices and downstream task evaluation scores.